In [ ]:
import networkx as nx

known_channels = set(df_profiles["channel_name"].values)

df_edges_filtered = df_edges[
    (df_edges["source"].isin(known_channels)) &
    (df_edges["target"].isin(known_channels))
].copy()

df_edges_agg = df_edges_filtered.groupby(["source", "target"])["weight"].sum().reset_index()

G = nx.DiGraph()
G.add_nodes_from(known_channels)

for _, row in df_edges_agg.iterrows():
    G.add_edge(row["source"], row["target"], weight=row["weight"])

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.6f}")

In [ ]:
pagerank = nx.pagerank(G, weight="weight")
in_deg = dict(G.in_degree(weight="weight"))
out_deg = dict(G.out_degree(weight="weight"))
betweenness = nx.betweenness_centrality(G, weight="weight", k=min(500, G.number_of_nodes()))

df_profiles["pagerank"] = df_profiles["channel_name"].map(pagerank).fillna(0)
df_profiles["in_degree_w"] = df_profiles["channel_name"].map(in_deg).fillna(0)
df_profiles["out_degree_w"] = df_profiles["channel_name"].map(out_deg).fillna(0)
df_profiles["betweenness"] = df_profiles["channel_name"].map(betweenness).fillna(0)

print("Top 10 PageRank:")
print(df_profiles.nlargest(10, "pagerank")[["channel_name", "cluster", "pagerank"]].to_string(index=False))

In [ ]:
from networkx.algorithms.community import louvain_communities

G_undirected = G.to_undirected()
communities = louvain_communities(G_undirected, weight="weight", resolution=1.0, seed=42)

community_map = {}
for idx, comm in enumerate(communities):
    for node in comm:
        community_map[node] = idx

df_profiles["louvain_community"] = df_profiles["channel_name"].map(community_map).fillna(-1).astype(int)

print(f"Communities found: {len(communities)}")
print(df_profiles["louvain_community"].value_counts().head(10))

In [ ]:
cluster_map = dict(zip(df_profiles["channel_name"], df_profiles["cluster"]))

interaction = np.zeros((K, K))

for _, row in df_edges_agg.iterrows():
    src_c = cluster_map.get(row["source"])
    tgt_c = cluster_map.get(row["target"])
    if src_c is not None and tgt_c is not None:
        interaction[int(src_c), int(tgt_c)] += row["weight"]

plt.figure(figsize=(8, 6))
sns.heatmap(interaction, annot=True, fmt=".0f", cmap="YlOrRd",
            xticklabels=[f"Cluster {i}" for i in range(K)],
            yticklabels=[f"Cluster {i}" for i in range(K)])
plt.xlabel("Target archetype (receives forward)")
plt.ylabel("Source archetype (sends forward)")
plt.title("Forwarding flow between archetypes")
plt.tight_layout()
plt.show()

In [ ]:
top_nodes = df_profiles.nlargest(200, "pagerank")["channel_name"].tolist()
G_sub = G.subgraph(top_nodes).copy()

node_colors = [cluster_map.get(n, -1) for n in G_sub.nodes()]
node_sizes = [pagerank.get(n, 0) * 50000 for n in G_sub.nodes()]

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G_sub, k=0.3, seed=42, weight="weight")
nx.draw_networkx_nodes(G_sub, pos, node_color=node_colors, cmap=plt.cm.tab10,
                       node_size=node_sizes, alpha=0.7)
nx.draw_networkx_edges(G_sub, pos, alpha=0.1, arrows=True, arrowsize=5)
plt.title("Top 200 channels by PageRank (color = archetype)")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import FancyBboxPatch

radar_cols = [
    "msgs_per_day", "avg_views", "forward_ratio", "frac_is_forward",
    "avg_toxicity", "frac_text_only", "avg_word_count", "n_fwd_sources"
]

means = df_profiles.groupby("cluster")[radar_cols].mean()
means_norm = (means - means.min()) / (means.max() - means.min() + 1e-9)

angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for cluster_id in range(K):
    values = means_norm.loc[cluster_id].tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", label=f"Cluster {cluster_id}", linewidth=2)
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_cols, size=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.title("Archetype profiles")
plt.tight_layout()
plt.show()

In [ ]:
df_profiles.to_csv("channel_profiles_final.csv", index=False)
df_edges_agg.to_csv("forwarding_edges_agg.csv", index=False)
cluster_means.T.to_csv("cluster_centroids.csv")

pd.DataFrame({
    "channel_name": list(pagerank.keys()),
    "pagerank": list(pagerank.values())
}).to_csv("pagerank_scores.csv", index=False)

print("Saved:")
print("  channel_profiles_final.csv")
print("  forwarding_edges_agg.csv")
print("  cluster_centroids.csv")
print("  pagerank_scores.csv")